# 115_paper_figures_2_14

Article-mode figures for SSRN 5005047 (core Figure 1..12) plus supplementary diagnostics.

Deterministic/scenario figures in this notebook start from policy SSS states (article-style initialization).

Outputs are written with an nb115-specific prefix to avoid collisions with strict-author notebook artifacts.

In [ ]:
import os, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch


def _find_project_root():
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir():
        return cand
    raise RuntimeError("Could not find project root containing src/.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelParams
from src.io_utils import load_json
from src.metrics import output_gap_from_consumption
from src.table2_builder import _load_run_dir, _load_net_from_run, _load_sim_paths
from src.steady_states import solve_flexprice_sss, solve_taylor_sss, solve_efficient_sss
from src.sss_from_policy import switching_policy_sss_by_regime_from_policy
from src.experiments import DeterministicPathSpec, simulate_deterministic_path

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
FIG_DIR = str(PROJECT_ROOT / "artifacts" / "paper_figures_nb115_article_sss")
FIG_PREFIX = "nb115_article_sss_"
os.makedirs(FIG_DIR, exist_ok=True)
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# paper-like defaults
PRE = 5
N_POST = 20
IR_H = 120
IR_SHOCK_MULT = 1.0   # author IR script includes ?1? and ?3? batches; here default is +1?
SHOCK_SIZE_MULT = 1.5 # Figure 13 larger persistent-shock case

print("Compute device:", COMPUTE_DEVICE)
print("Figure output dir:", FIG_DIR)
print("Figure file prefix:", FIG_PREFIX)

ann = lambda x: 400.0 * np.asarray(x)


def _parse_dtype(s):
    if s is None:
        return torch.float64
    s = str(s)
    if "float32" in s:
        return torch.float32
    if "float64" in s:
        return torch.float64
    return torch.float64


def load_params_from_run(run_dir: str, *, device: str | None = None) -> ModelParams:
    cfg = load_json(os.path.join(run_dir, "config.json"))
    p = cfg.get("params", {})
    keep = {k:v for k,v in p.items() if k in ModelParams.__dataclass_fields__}
    keep["device"] = device or COMPUTE_DEVICE
    keep["dtype"] = _parse_dtype(p.get("dtype", "float64"))
    return ModelParams(**keep).to_torch()


def copy_params(p: ModelParams, **overrides) -> ModelParams:
    d = {k:getattr(p,k) for k in ModelParams.__dataclass_fields__.keys()}
    d.update(overrides)
    return ModelParams(**d).to_torch()


def load_policy(policy: str):
    run_dir = _load_run_dir(ARTIFACTS_ROOT, policy, use_selected=True)
    params = load_params_from_run(run_dir, device=COMPUTE_DEVICE)
    net = _load_net_from_run(run_dir, params, policy)
    sim = None
    try:
        sim = _load_sim_paths(run_dir)
    except Exception:
        pass
    return run_dir, params, net, sim


def make_rbar(params: ModelParams) -> torch.Tensor:
    flex = solve_flexprice_sss(params)
    return torch.tensor([flex.by_regime[0]["r_star"], flex.by_regime[1]["r_star"]], device=params.device, dtype=params.dtype)


def sss_by_regime(params: ModelParams, policy: str, net):
    rbar = make_rbar(params) if policy in ("mod_taylor", "mod_taylor_zlb") else None
    return switching_policy_sss_by_regime_from_policy(params, net, policy=policy, rbar_by_regime=rbar).by_regime


def x0_from_sss(policy: str, sss: dict, regime: int, *, dtype=torch.float32):
    b = sss[int(regime)]
    if policy == "commitment":
        return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime),
                              float(b["vartheta_prev"]), float(b["varrho_prev"]), float(b.get("c_prev", b["c"]))]], dtype=dtype)
    if policy == "commitment_zlb":
        return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime),
                              float(b["vartheta_prev"]), float(b["varrho_prev"]), float(b.get("c_prev", b["c"])),
                              float(b.get("i_nom_prev", b.get("i_nom", 0.0))), float(b.get("varphi_prev", b.get("varphi", 0.0)))]], dtype=dtype)
    return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime)]], dtype=dtype)


def ensure_same_calibration(params_a: ModelParams, params_b: ModelParams, *, atol: float = 1e-12):
    for k in ModelParams.__dataclass_fields__.keys():
        if k in ("device", "dtype"):
            continue
        va = float(getattr(params_a, k))
        vb = float(getattr(params_b, k))
        if abs(va - vb) > atol:
            raise RuntimeError(f"Calibration mismatch on {k}: {va} vs {vb}")


def series_from_sim(sim: dict, params: ModelParams):
    s = np.asarray(sim["s"]).reshape(-1).astype(np.int64)
    pi = np.asarray(sim["pi"]).reshape(-1)
    i_nom = np.asarray(sim["i"]).reshape(-1)
    x_gap = output_gap_from_consumption(sim, solve_efficient_sss(params), params=params, time_varying=True)
    r = i_nom - pi
    return {
        "s": s,
        "pi": pi,
        "i": i_nom,
        "x": np.asarray(x_gap).reshape(-1),
        "r": np.asarray(r).reshape(-1),
        "s_r": s,
        "Delta": np.asarray(sim["Delta"]).reshape(-1),
    }


def savefig(name: str):
    filename = f"{FIG_PREFIX}{name}.png"
    p = os.path.join(FIG_DIR, filename)
    plt.savefig(p, dpi=160, bbox_inches="tight")
    print("Saved:", p)

## Load policies/runs

In [ ]:
run_t, p_t, net_t, sim_t = load_policy("taylor")
run_m, p_m, net_m, sim_m = load_policy("mod_taylor")
run_d, p_d, net_d, sim_d = load_policy("discretion")
run_c, p_c, net_c, sim_c = load_policy("commitment")

# Optional ZLB runs (required for Fig 10/11/14)
zlb_ok = True
try:
    run_tz, p_tz, net_tz, sim_tz = load_policy("taylor_zlb")
    run_mz, p_mz, net_mz, sim_mz = load_policy("mod_taylor_zlb")
    run_dz, p_dz, net_dz, sim_dz = load_policy("discretion_zlb")
    run_cz, p_cz, net_cz, sim_cz = load_policy("commitment_zlb")
except Exception as e:
    zlb_ok = False
    print("ZLB runs unavailable:", e)

print("zlb_ok:", zlb_ok)


## Figure 9: Ergodic distribution (Taylor vs Modified Taylor)

In [ ]:
ensure_same_calibration(p_t, p_m)
if sim_t is None or sim_m is None:
    raise RuntimeError("Figure 2 requires sim_paths for taylor and mod_taylor")

ser_t = series_from_sim(sim_t, p_t)
ser_m = series_from_sim(sim_m, p_t)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0,0].hist(ann(ser_t["pi"])[ser_t["s"]==0], bins=60, alpha=0.45, label="Taylor normal", color="tab:blue")
ax[0,0].hist(ann(ser_t["pi"])[ser_t["s"]==1], bins=60, alpha=0.45, label="Taylor bad", color="tab:orange")
ax[0,0].hist(ann(ser_m["pi"])[ser_m["s"]==0], bins=60, alpha=0.45, label="Mod Taylor normal", color="tab:green")
ax[0,0].hist(ann(ser_m["pi"])[ser_m["s"]==1], bins=60, alpha=0.45, label="Mod Taylor bad", color="tab:red")
ax[0,0].set_title("(a) Inflation")
ax[0,0].set_xlabel("Ann. perc.")
ax[0,0].legend(fontsize=8)

ax[0,1].hist(100*ser_t["x"][ser_t["s"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[0,1].hist(100*ser_t["x"][ser_t["s"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[0,1].hist(100*ser_m["x"][ser_m["s"]==0], bins=60, alpha=0.45, color="tab:green")
ax[0,1].hist(100*ser_m["x"][ser_m["s"]==1], bins=60, alpha=0.45, color="tab:red")
ax[0,1].set_title("(b) Output gap")
ax[0,1].set_xlabel("Perc. of log")

ax[1,0].hist(ann(ser_t["i"])[ser_t["s"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[1,0].hist(ann(ser_t["i"])[ser_t["s"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[1,0].hist(ann(ser_m["i"])[ser_m["s"]==0], bins=60, alpha=0.45, color="tab:green")
ax[1,0].hist(ann(ser_m["i"])[ser_m["s"]==1], bins=60, alpha=0.45, color="tab:red")
ax[1,0].set_title("(c) Nominal interest rate")
ax[1,0].set_xlabel("Ann. perc.")

ax[1,1].hist(ann(ser_t["r"])[ser_t["s_r"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[1,1].hist(ann(ser_t["r"])[ser_t["s_r"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[1,1].hist(ann(ser_m["r"])[ser_m["s_r"]==0], bins=60, alpha=0.45, color="tab:green")
ax[1,1].hist(ann(ser_m["r"])[ser_m["s_r"]==1], bins=60, alpha=0.45, color="tab:red")
ax[1,1].set_title("(d) Real interest rate")
ax[1,1].set_xlabel("Ann. perc.")

fig.suptitle("Figure 2: Ergodic distribution", y=1.02)
plt.tight_layout(); savefig("figure2"); plt.show()


## Supplementary: Response to a regime change (Taylor vs Modified Taylor)

In [ ]:
sss_t = sss_by_regime(p_t, "taylor", net_t)
sss_m = sss_by_regime(p_m, "mod_taylor", net_m)

x0_t = x0_from_sss("taylor", sss_t, 0)
x0_m = x0_from_sss("mod_taylor", sss_m, 0)

# Innovations to temporary shocks are set to zero; start at normal SSS.
T = N_POST + 1
spec = DeterministicPathSpec(T=T, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0] + [1]*T)

path_t = simulate_deterministic_path(p_t, "taylor", net_t, x0=x0_t, spec=spec, compute_implied_i=True)
path_m = simulate_deterministic_path(p_m, "mod_taylor", net_m, x0=x0_m, spec=spec, rbar_by_regime=make_rbar(p_m), compute_implied_i=True)

def transition_arrays(path, params):
    eff = solve_efficient_sss(params)
    pi = path["pi"][:,0]; i = path["i"][:,0]; D = path["Delta"][:,0]
    x = output_gap_from_consumption(path, eff, params=params, time_varying=True)
    r = i - pi
    pi = np.concatenate([np.full(PRE, pi[0]), pi[1:1+N_POST]])
    x = np.concatenate([np.full(PRE, x[0]), x[1:1+N_POST]])
    r = np.concatenate([np.full(PRE, r[1]), r[1:1+N_POST]])
    D = np.concatenate([np.full(PRE, D[0]), D[1:1+N_POST]])
    return pi, x, r, D

pi_ta, x_ta, r_ta, D_ta = transition_arrays(path_t, p_t)
pi_ma, x_ma, r_ma, D_ma = transition_arrays(path_m, p_m)

t = np.arange(-PRE, N_POST)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_ta), label='Taylor')
ax[0,0].plot(t, ann(pi_ma), '--', label='Modified Taylor')
ax[0,0].axhline(0,color='k',lw=1)
ax[0,0].set_title('(a) Inflation')
ax[0,1].plot(t, 100*x_ta, label='Taylor')
ax[0,1].plot(t, 100*x_ma, '--', label='Modified Taylor')
ax[0,1].axhline(0,color='k',lw=1)
ax[0,1].set_title('(b) Output gap')
ax[1,0].plot(t, ann(r_ta), label='Taylor')
ax[1,0].plot(t, ann(r_ma), '--', label='Modified Taylor')
ax[1,0].axhline(0,color='k',lw=1)
ax[1,0].set_title('(c) Real interest rate')
ax[1,1].plot(t, D_ta, label='Taylor')
ax[1,1].plot(t, D_ma, '--', label='Modified Taylor')
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel():
    a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 3: Response to a regime change (Taylor rule)', y=1.02)
plt.tight_layout(); savefig('figure3'); plt.show()


## Figure 10: Sensitivity to regime length (Taylor rule)

In [ ]:
params0 = ModelParams(device=COMPUTE_DEVICE, dtype=torch.float64).to_torch()
grid_p21 = np.linspace(0.02, 0.98, 40)
dur_bad = 1.0 / grid_p21

pi_normal = []; r_normal = []; pi_bad_cf = []; r_bad_cf = []
for p21 in grid_p21:
    p_base = copy_params(params0, p12=float(params0.p12), p21=float(p21))
    flex_b = solve_flexprice_sss(p_base)
    t_b = solve_taylor_sss(p_base, flex_b)
    pi_normal.append(float(t_b.by_regime[0]['pi']))
    r_normal.append(float(t_b.by_regime[0]['r']))

    p_cf = copy_params(params0, p12=1.0, p21=float(p21))
    flex_c = solve_flexprice_sss(p_cf)
    t_c = solve_taylor_sss(p_cf, flex_c)
    pi_bad_cf.append(float(t_c.by_regime[1]['pi']))
    r_bad_cf.append(float(t_c.by_regime[1]['r']))

eff = solve_efficient_sss(params0)
idx = np.argsort(dur_bad)
x = dur_bad[idx]

fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].plot(x, ann(np.array(pi_normal)[idx]), label='Baseline (normal SSS)')
ax[0].plot(x, ann(np.array(pi_bad_cf)[idx]), '--', label='Counterfactual p12->1 (bad SSS)')
ax[0].axhline(0.0, color='gray', ls=':')
ax[0].set_title('(a) Inflation')
ax[0].set_xlabel('Average bad-times duration (quarters)')
ax[0].set_ylabel('Ann. perc.')
ax[0].legend()

ax[1].plot(x, ann(np.array(r_normal)[idx]), label='Baseline (normal SSS)')
ax[1].plot(x, ann(np.array(r_bad_cf)[idx]), '--', label='Counterfactual p12->1 (bad SSS)')
ax[1].axhline(ann(float(eff['r_hat'])), color='gray', ls=':')
ax[1].set_title('(b) Real interest rate')
ax[1].set_xlabel('Average bad-times duration (quarters)')
ax[1].set_ylabel('Ann. perc.')
ax[1].legend()

fig.suptitle('Figure 4: Sensitivity to regime length (Taylor rule)', y=1.03)
plt.tight_layout(); savefig('figure4'); plt.show()


## Figure 8: Ergodic distribution (Discretion)

In [ ]:
if sim_d is None:
    raise RuntimeError('Figure 5 requires discretion sim_paths.npz')
ser = series_from_sim(sim_d, p_d)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].hist(ann(ser['pi'])[ser['s']==0], bins=60, alpha=0.6, label='normal (s=0)')
ax[0,0].hist(ann(ser['pi'])[ser['s']==1], bins=60, alpha=0.6, label='bad (s=1)')
ax[0,0].set_title('(a) Inflation'); ax[0,0].legend()
ax[0,1].hist(100*ser['x'][ser['s']==0], bins=60, alpha=0.6)
ax[0,1].hist(100*ser['x'][ser['s']==1], bins=60, alpha=0.6)
ax[0,1].set_title('(b) Output gap')
ax[1,0].hist(ann(ser['r'])[ser['s_r']==0], bins=60, alpha=0.6)
ax[1,0].hist(ann(ser['r'])[ser['s_r']==1], bins=60, alpha=0.6)
ax[1,0].set_title('(c) Real interest rate')
ax[1,1].hist(ser['Delta'][ser['s']==0], bins=60, alpha=0.6)
ax[1,1].hist(ser['Delta'][ser['s']==1], bins=60, alpha=0.6)
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel(): a.set_xlabel('model units')
fig.suptitle('Figure 5: Ergodic distribution: discretion', y=1.02)
plt.tight_layout(); savefig('figure5'); plt.show()


## Figure 1: Ergodic distribution (Commitment)

In [ ]:
if sim_c is None:
    raise RuntimeError('Figure 6 requires commitment sim_paths.npz')
ser = series_from_sim(sim_c, p_c)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].hist(ann(ser['pi'])[ser['s']==0], bins=60, alpha=0.6, label='normal (s=0)')
ax[0,0].hist(ann(ser['pi'])[ser['s']==1], bins=60, alpha=0.6, label='bad (s=1)')
ax[0,0].set_title('(a) Inflation'); ax[0,0].legend()
ax[0,1].hist(100*ser['x'][ser['s']==0], bins=60, alpha=0.6)
ax[0,1].hist(100*ser['x'][ser['s']==1], bins=60, alpha=0.6)
ax[0,1].set_title('(b) Output gap')
ax[1,0].hist(ann(ser['r'])[ser['s_r']==0], bins=60, alpha=0.6)
ax[1,0].hist(ann(ser['r'])[ser['s_r']==1], bins=60, alpha=0.6)
ax[1,0].set_title('(c) Real interest rate')
ax[1,1].hist(ser['Delta'][ser['s']==0], bins=60, alpha=0.6)
ax[1,1].hist(ser['Delta'][ser['s']==1], bins=60, alpha=0.6)
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel(): a.set_xlabel('model units')
fig.suptitle('Figure 6: Ergodic distribution: commitment', y=1.02)
plt.tight_layout(); savefig('figure6'); plt.show()


## Supplementary: Temporary AR(1) cost-push shock in bad regime (Commitment vs Modified Taylor)

In [ ]:
ensure_same_calibration(p_c, p_m)
sss_c = sss_by_regime(p_c, 'commitment', net_c)
sss_m = sss_by_regime(p_m, 'mod_taylor', net_m)

x0_cb = x0_from_sss('commitment', sss_c, 1)
x0_mb = x0_from_sss('mod_taylor', sss_m, 1)

xi_jump_c = IR_SHOCK_MULT * float(p_c.sigma_tau)
xi_jump_m = IR_SHOCK_MULT * float(p_m.sigma_tau)
x0_cb[:,3] += xi_jump_c
x0_mb[:,3] += xi_jump_m

spec_bad = DeterministicPathSpec(T=IR_H, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[1]*(IR_H+1))
path_cb = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_cb, spec=spec_bad, compute_implied_i=True)
path_mb = simulate_deterministic_path(p_m, 'mod_taylor', net_m, x0=x0_mb, spec=spec_bad, rbar_by_regime=make_rbar(p_m), compute_implied_i=True)


def irf_arrays(path, params):
    eff = solve_efficient_sss(params)
    pi = path['pi'][:,0]
    i = path['i'][:,0]
    x = output_gap_from_consumption(path, eff, params=params, time_varying=True)
    r = (i - pi)[:-1]
    P = np.cumprod(1.0 + pi)
    return pi, x, r, P

pi_c, x_c, r_c, P_c = irf_arrays(path_cb, p_c)
pi_m2, x_m2, r_m2, P_m2 = irf_arrays(path_mb, p_m)

t = np.arange(len(pi_c))
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_c), label='Commitment')
ax[0,0].plot(t, ann(pi_m2), '--', label='Modified Taylor')
ax[0,0].axhline(0,color='k',lw=1)
ax[0,0].set_title('(a) Inflation')
ax[0,1].plot(t, 100*x_c, label='Commitment')
ax[0,1].plot(t, 100*x_m2, '--', label='Modified Taylor')
ax[0,1].axhline(0,color='k',lw=1)
ax[0,1].set_title('(b) Output gap')
ax[1,0].plot(t[:-1], ann(r_c), label='Commitment')
ax[1,0].plot(t[:-1], ann(r_m2), '--', label='Modified Taylor')
ax[1,0].axhline(0,color='k',lw=1)
ax[1,0].set_title('(c) Real interest rate')
ax[1,1].plot(t, P_c, label='Commitment')
ax[1,1].plot(t, P_m2, '--', label='Modified Taylor')
ax[1,1].set_title('(d) Price level')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 7: IRF to transitory cost-push shock', y=1.02)
plt.tight_layout(); savefig('figure7'); plt.show()


## Figure 2: Regime change (Commitment vs Discretion)

In [ ]:
ensure_same_calibration(p_c, p_d)
sss_c = sss_by_regime(p_c, 'commitment', net_c)
sss_d = sss_by_regime(p_d, 'discretion', net_d)
x0_c = x0_from_sss('commitment', sss_c, 0)
x0_d = x0_from_sss('discretion', sss_d, 0)

spec_sw = DeterministicPathSpec(T=N_POST+1, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0]+[1]*(N_POST+1))
path_c = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_c, spec=spec_sw, compute_implied_i=True)
path_d = simulate_deterministic_path(p_d, 'discretion', net_d, x0=x0_d, spec=spec_sw, compute_implied_i=True)


def transition_pack(path, params):
    eff = solve_efficient_sss(params)
    pi = path['pi'][:,0]; i = path['i'][:,0]; D = path['Delta'][:,0]
    x = output_gap_from_consumption(path, eff, params=params, time_varying=True)
    r = i - pi
    pi = np.concatenate([np.full(PRE, pi[0]), pi[1:1+N_POST]])
    x = np.concatenate([np.full(PRE, x[0]), x[1:1+N_POST]])
    r = np.concatenate([np.full(PRE, r[1]), r[1:1+N_POST]])
    D = np.concatenate([np.full(PRE, D[0]), D[1:1+N_POST]])
    return pi, x, r, D

pi_c2, x_c2, r_c2, D_c2 = transition_pack(path_c, p_c)
pi_d2, x_d2, r_d2, D_d2 = transition_pack(path_d, p_d)

t = np.arange(-PRE, N_POST)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_c2), label='Commitment')
ax[0,0].plot(t, ann(pi_d2), '--', label='Discretion')
ax[0,0].axhline(0,color='k',lw=1)
ax[0,0].set_title('(a) Inflation')
ax[0,1].plot(t, 100*x_c2, label='Commitment')
ax[0,1].plot(t, 100*x_d2, '--', label='Discretion')
ax[0,1].axhline(0,color='k',lw=1)
ax[0,1].set_title('(b) Output gap')
ax[1,0].plot(t, ann(r_c2), label='Commitment')
ax[1,0].plot(t, ann(r_d2), '--', label='Discretion')
ax[1,0].axhline(0,color='k',lw=1)
ax[1,0].set_title('(c) Real interest rate')
ax[1,1].plot(t, D_c2, label='Commitment')
ax[1,1].plot(t, D_d2, '--', label='Discretion')
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 8: Response to a regime change', y=1.02)
plt.tight_layout(); savefig('figure8'); plt.show()


## Figure 4: IRF under commitment, baseline vs higher persistence

In [ ]:
# paper note: in a model without persistent supply shocks
p_base = copy_params(p_c, eta_bar=0.0, p12=0.0, p21=1.0)
p_hi = copy_params(p_base, rho_tau=0.99)

sss_base = switching_policy_sss_by_regime_from_policy(p_base, net_c, policy='commitment').by_regime
x0_base = x0_from_sss('commitment', sss_base, 0)
x0_base[:,3] += IR_SHOCK_MULT * float(p_base.sigma_tau)

spec_n = DeterministicPathSpec(T=IR_H, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0]*(IR_H+1))
path_base = simulate_deterministic_path(p_base, 'commitment', net_c, x0=x0_base, spec=spec_n, compute_implied_i=True)
path_hi = simulate_deterministic_path(p_hi, 'commitment', net_c, x0=x0_base, spec=spec_n, compute_implied_i=True)

pi_b, x_b, r_b, P_b = irf_arrays(path_base, p_base)
pi_h, x_h, r_h, P_h = irf_arrays(path_hi, p_hi)

t = np.arange(len(pi_b))
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_b), label='baseline rho_tau')
ax[0,0].plot(t, ann(pi_h), '--', label='rho_tau=0.99')
ax[0,0].set_title('(a) Inflation'); ax[0,0].axhline(0,color='k',lw=1)
ax[0,1].plot(t, 100*x_b, label='baseline rho_tau')
ax[0,1].plot(t, 100*x_h, '--', label='rho_tau=0.99')
ax[0,1].set_title('(b) Output gap'); ax[0,1].axhline(0,color='k',lw=1)
ax[1,0].plot(t[:-1], ann(r_b), label='baseline rho_tau')
ax[1,0].plot(t[:-1], ann(r_h), '--', label='rho_tau=0.99')
ax[1,0].set_title('(c) Real interest rate'); ax[1,0].axhline(0,color='k',lw=1)
ax[1,1].plot(t, P_b, label='baseline rho_tau')
ax[1,1].plot(t, P_h, '--', label='rho_tau=0.99')
ax[1,1].set_title('(d) Price level')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 9: IRF with different persistence', y=1.02)
plt.tight_layout(); savefig('figure9'); plt.show()


## Supplementary ZLB: Ergodic distribution (Taylor vs Modified Taylor)

In [ ]:
if not zlb_ok:
    raise RuntimeError('Figure 10 requires taylor_zlb and mod_taylor_zlb runs (not available).')
ensure_same_calibration(p_tz, p_mz)
if sim_tz is None or sim_mz is None:
    raise RuntimeError('Figure 10 requires sim_paths for taylor_zlb and mod_taylor_zlb')

ser_tz = series_from_sim(sim_tz, p_tz)
ser_mz = series_from_sim(sim_mz, p_tz)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0,0].hist(ann(ser_tz['pi'])[ser_tz['s']==0], bins=60, alpha=0.45, color='tab:blue', label='Taylor ZLB normal')
ax[0,0].hist(ann(ser_tz['pi'])[ser_tz['s']==1], bins=60, alpha=0.45, color='tab:orange', label='Taylor ZLB bad')
ax[0,0].hist(ann(ser_mz['pi'])[ser_mz['s']==0], bins=60, alpha=0.45, color='tab:green', label='Mod Taylor ZLB normal')
ax[0,0].hist(ann(ser_mz['pi'])[ser_mz['s']==1], bins=60, alpha=0.45, color='tab:red', label='Mod Taylor ZLB bad')
ax[0,0].set_title('(a) Inflation'); ax[0,0].legend(fontsize=8)
ax[0,1].hist(100*ser_tz['x'][ser_tz['s']==0], bins=60, alpha=0.45, color='tab:blue')
ax[0,1].hist(100*ser_tz['x'][ser_tz['s']==1], bins=60, alpha=0.45, color='tab:orange')
ax[0,1].hist(100*ser_mz['x'][ser_mz['s']==0], bins=60, alpha=0.45, color='tab:green')
ax[0,1].hist(100*ser_mz['x'][ser_mz['s']==1], bins=60, alpha=0.45, color='tab:red')
ax[0,1].set_title('(b) Output gap')
ax[1,0].hist(ann(ser_tz['i'])[ser_tz['s']==0], bins=60, alpha=0.45, color='tab:blue')
ax[1,0].hist(ann(ser_tz['i'])[ser_tz['s']==1], bins=60, alpha=0.45, color='tab:orange')
ax[1,0].hist(ann(ser_mz['i'])[ser_mz['s']==0], bins=60, alpha=0.45, color='tab:green')
ax[1,0].hist(ann(ser_mz['i'])[ser_mz['s']==1], bins=60, alpha=0.45, color='tab:red')
ax[1,0].set_title('(c) Nominal interest rate')
ax[1,1].hist(ann(ser_tz['r'])[ser_tz['s_r']==0], bins=60, alpha=0.45, color='tab:blue')
ax[1,1].hist(ann(ser_tz['r'])[ser_tz['s_r']==1], bins=60, alpha=0.45, color='tab:orange')
ax[1,1].hist(ann(ser_mz['r'])[ser_mz['s_r']==0], bins=60, alpha=0.45, color='tab:green')
ax[1,1].hist(ann(ser_mz['r'])[ser_mz['s_r']==1], bins=60, alpha=0.45, color='tab:red')
ax[1,1].set_title('(d) Real interest rate')
for a in ax.ravel(): a.set_xlabel('model units')
fig.suptitle('Figure 10: ZLB ergodic distribution: Taylor rules', y=1.02)
plt.tight_layout(); savefig('figure10'); plt.show()


## Supplementary ZLB: Regime change under commitment (with vs without ZLB)

In [ ]:
if not zlb_ok:
    raise RuntimeError('Figure 11 requires commitment_zlb run (not available).')
sss_c = sss_by_regime(p_c, 'commitment', net_c)
sss_cz = sss_by_regime(p_cz, 'commitment_zlb', net_cz)
x0_c = x0_from_sss('commitment', sss_c, 0)
x0_cz = x0_from_sss('commitment_zlb', sss_cz, 0)

spec_sw = DeterministicPathSpec(T=N_POST+1, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0]+[1]*(N_POST+1))
path_c = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_c, spec=spec_sw, compute_implied_i=True)
path_cz = simulate_deterministic_path(p_cz, 'commitment_zlb', net_cz, x0=x0_cz, spec=spec_sw, compute_implied_i=True)

pi_c2, x_c2, r_c2, D_c2 = transition_pack(path_c, p_c)
pi_z2, x_z2, r_z2, D_z2 = transition_pack(path_cz, p_cz)

t = np.arange(-PRE, N_POST)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_c2), label='Commitment (no ZLB)')
ax[0,0].plot(t, ann(pi_z2), '--', label='Commitment ZLB')
ax[0,0].set_title('(a) Inflation'); ax[0,0].axhline(0,color='k',lw=1)
ax[0,1].plot(t, 100*x_c2, label='Commitment (no ZLB)')
ax[0,1].plot(t, 100*x_z2, '--', label='Commitment ZLB')
ax[0,1].set_title('(b) Output gap'); ax[0,1].axhline(0,color='k',lw=1)
ax[1,0].plot(t, ann(r_c2), label='Commitment (no ZLB)')
ax[1,0].plot(t, ann(r_z2), '--', label='Commitment ZLB')
ax[1,0].set_title('(c) Real interest rate'); ax[1,0].axhline(0,color='k',lw=1)
ax[1,1].plot(t, D_c2, label='Commitment (no ZLB)')
ax[1,1].plot(t, D_z2, '--', label='Commitment ZLB')
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 11: ZLB regime change under commitment', y=1.02)
plt.tight_layout(); savefig('figure11'); plt.show()


## Figure 7: Price level dynamics under commitment (10,000 quarters)

In [ ]:
if sim_c is None:
    raise RuntimeError('Figure 12 requires commitment sim_paths.npz')
pi_sim = np.asarray(sim_c['pi']).reshape(-1)
T_show = min(10000, len(pi_sim))
P = np.cumprod(1.0 + pi_sim[:T_show])

fig, ax = plt.subplots(1,1,figsize=(11,4))
ax.plot(np.arange(T_show), P)
ax.set_title('Figure 12: Price level dynamics under commitment')
ax.set_xlabel('Quarter')
ax.set_ylabel('Price level index')
plt.tight_layout(); savefig('figure12'); plt.show()


## Figure 12: Regime change, baseline vs larger persistent supply shock

In [ ]:
p_big = copy_params(p_c, eta_bar=float(p_c.eta_bar) * SHOCK_SIZE_MULT)
sss_base = sss_by_regime(p_c, 'commitment', net_c)
sss_big = switching_policy_sss_by_regime_from_policy(p_big, net_c, policy='commitment').by_regime

x0_base = x0_from_sss('commitment', sss_base, 0)
x0_big = x0_from_sss('commitment', sss_big, 0)

spec_sw2 = DeterministicPathSpec(T=N_POST+1, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0]+[1]*(N_POST+1))
path_base = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_base, spec=spec_sw2, compute_implied_i=True)
path_big = simulate_deterministic_path(p_big, 'commitment', net_c, x0=x0_big, spec=spec_sw2, compute_implied_i=True)

pi_b2, x_b2, r_b2, D_b2 = transition_pack(path_base, p_c)
pi_g2, x_g2, r_g2, D_g2 = transition_pack(path_big, p_big)

t = np.arange(-PRE, N_POST)
fig, ax = plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t, ann(pi_b2), label='baseline')
ax[0,0].plot(t, ann(pi_g2), '--', label='larger shock')
ax[0,0].set_title('(a) Inflation'); ax[0,0].axhline(0,color='k',lw=1)
ax[0,1].plot(t, 100*x_b2, label='baseline')
ax[0,1].plot(t, 100*x_g2, '--', label='larger shock')
ax[0,1].set_title('(b) Output gap'); ax[0,1].axhline(0,color='k',lw=1)
ax[1,0].plot(t, ann(r_b2), label='baseline')
ax[1,0].plot(t, ann(r_g2), '--', label='larger shock')
ax[1,0].set_title('(c) Real interest rate'); ax[1,0].axhline(0,color='k',lw=1)
ax[1,1].plot(t, D_b2, label='baseline')
ax[1,1].plot(t, D_g2, '--', label='larger shock')
ax[1,1].set_title('(d) Price dispersion')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend()
fig.suptitle('Figure 13: Response to a regime change: shock size', y=1.02)
plt.tight_layout(); savefig('figure13'); plt.show()


## Supplementary ZLB: Ergodic distribution (Discretion vs Commitment)

In [ ]:
if not zlb_ok:
    raise RuntimeError('Figure 14 requires discretion_zlb and commitment_zlb runs (not available).')
ensure_same_calibration(p_dz, p_cz)
if sim_dz is None or sim_cz is None:
    raise RuntimeError('Figure 14 requires sim_paths for discretion_zlb and commitment_zlb')

ser_dz = series_from_sim(sim_dz, p_dz)
ser_cz = series_from_sim(sim_cz, p_dz)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0,0].hist(ann(ser_dz['pi'])[ser_dz['s']==0], bins=60, alpha=0.45, color='tab:blue', label='Discretion ZLB normal')
ax[0,0].hist(ann(ser_dz['pi'])[ser_dz['s']==1], bins=60, alpha=0.45, color='tab:orange', label='Discretion ZLB bad')
ax[0,0].hist(ann(ser_cz['pi'])[ser_cz['s']==0], bins=60, alpha=0.45, color='tab:green', label='Commitment ZLB normal')
ax[0,0].hist(ann(ser_cz['pi'])[ser_cz['s']==1], bins=60, alpha=0.45, color='tab:red', label='Commitment ZLB bad')
ax[0,0].set_title('(a) Inflation'); ax[0,0].legend(fontsize=8)
ax[0,1].hist(100*ser_dz['x'][ser_dz['s']==0], bins=60, alpha=0.45, color='tab:blue')
ax[0,1].hist(100*ser_dz['x'][ser_dz['s']==1], bins=60, alpha=0.45, color='tab:orange')
ax[0,1].hist(100*ser_cz['x'][ser_cz['s']==0], bins=60, alpha=0.45, color='tab:green')
ax[0,1].hist(100*ser_cz['x'][ser_cz['s']==1], bins=60, alpha=0.45, color='tab:red')
ax[0,1].set_title('(b) Output gap')
ax[1,0].hist(ann(ser_dz['i'])[ser_dz['s']==0], bins=60, alpha=0.45, color='tab:blue')
ax[1,0].hist(ann(ser_dz['i'])[ser_dz['s']==1], bins=60, alpha=0.45, color='tab:orange')
ax[1,0].hist(ann(ser_cz['i'])[ser_cz['s']==0], bins=60, alpha=0.45, color='tab:green')
ax[1,0].hist(ann(ser_cz['i'])[ser_cz['s']==1], bins=60, alpha=0.45, color='tab:red')
ax[1,0].set_title('(c) Nominal interest rate')
ax[1,1].hist(ann(ser_dz['r'])[ser_dz['s_r']==0], bins=60, alpha=0.45, color='tab:blue')
ax[1,1].hist(ann(ser_dz['r'])[ser_dz['s_r']==1], bins=60, alpha=0.45, color='tab:orange')
ax[1,1].hist(ann(ser_cz['r'])[ser_cz['s_r']==0], bins=60, alpha=0.45, color='tab:green')
ax[1,1].hist(ann(ser_cz['r'])[ser_cz['s_r']==1], bins=60, alpha=0.45, color='tab:red')
ax[1,1].set_title('(d) Real interest rate')
for a in ax.ravel(): a.set_xlabel('model units')
fig.suptitle('Figure 14: ZLB ergodic distribution', y=1.02)
plt.tight_layout(); savefig('figure14'); plt.show()


### Notes
- Figures 10/11/14 require completed ZLB runs from notebooks 14..17.
- Figure 7/9 use AR(1) cost-push shock via initial jump in `xi` and zero subsequent innovations.
- Figure 9 follows paper wording by setting `eta_bar=0` and fixed normal regime.